# `rag-stress` — T4 Batched & Resumable LLM Inference

This notebook runs the final `rag-stress` datasets through one fixed LLM configuration.

It supports:

- Clean baseline
- D1 / D3 / D5 distractor conditions
- Beginning / Middle / End position conditions
- Counterfactual condition
- vLLM batched inference
- NVIDIA T4-oriented settings
- Resumable JSONL outputs
- Fixed prompt across all conditions
- Short-answer generation only

## Do we need to save a vector database first?

**No, not for this stage.**

The perturbation datasets already contain the exact context that should be shown to the LLM.

So the experiment is now:

```text
Prepared perturbation JSONL
        ↓
Fixed prompt
        ↓
LLM
        ↓
Saved prediction
```

A FAISS/vector index only needs to be saved if you later want to reproduce retrieval, retrieve new passages, rebuild hard negatives, or evaluate retrieval quality itself.


## 1. Install dependencies

In [1]:
!pip -q install -U vllm transformers tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/4

## 2. Imports and configuration

In [1]:
from pathlib import Path
import json
from collections import Counter

import torch
from tqdm.auto import tqdm
from google.colab import drive


drive.mount("/content/drive")


CLEAN_FILE = Path("clean_500.jsonl")
DISTRACTOR_FILE = Path("distractor_D1_D3_D5_500.jsonl")
POSITION_FILE = Path("position_begin_middle_end_500.jsonl")
COUNTERFACTUAL_FILE = Path("counterfactual_high_confidence_fixed.jsonl")

RESULTS_DIR = Path("/content/drive/MyDrive/rag_stress/outputs")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_OUT = RESULTS_DIR / "clean_answers.jsonl"
DISTRACTOR_OUT = RESULTS_DIR / "distractor_answers.jsonl"
POSITION_OUT = RESULTS_DIR / "position_answers.jsonl"
COUNTERFACTUAL_OUT = RESULTS_DIR / "counterfactual_answers.jsonl"

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

GPU_MEMORY_UTILIZATION = 0.90
MAX_MODEL_LEN = 8192
INFERENCE_BATCH_SIZE = 64

MAX_NEW_TOKENS = 32
TEMPERATURE = 0.0
TOP_P = 1.0
SEED = 42

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

print("Model:", MODEL_NAME)

print("\nOutputs will be saved to:")
print(RESULTS_DIR)

print("\nOutput files:")
print("Clean          :", CLEAN_OUT)
print("Distractor     :", DISTRACTOR_OUT)
print("Position       :", POSITION_OUT)
print("Counterfactual :", COUNTERFACTUAL_OUT)

Mounted at /content/drive
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB
Model: Qwen/Qwen3-4B-Instruct-2507

Outputs will be saved to:
/content/drive/MyDrive/rag_stress/outputs

Output files:
Clean          : /content/drive/MyDrive/rag_stress/outputs/clean_answers.jsonl
Distractor     : /content/drive/MyDrive/rag_stress/outputs/distractor_answers.jsonl
Position       : /content/drive/MyDrive/rag_stress/outputs/position_answers.jsonl
Counterfactual : /content/drive/MyDrive/rag_stress/outputs/counterfactual_answers.jsonl


## 3. Validate dataset files

In [2]:
files = {
    "clean": CLEAN_FILE,
    "distractor": DISTRACTOR_FILE,
    "position": POSITION_FILE,
    "counterfactual": COUNTERFACTUAL_FILE,
}

for name, path in files.items():
    print(f"{name:15s}", "FOUND" if path.exists() else "MISSING", "->", path)

missing = [str(path) for path in files.values() if not path.exists()]
assert not missing, "Missing dataset file(s):\n" + "\n".join(missing)

clean           FOUND -> clean_500.jsonl
distractor      FOUND -> distractor_D1_D3_D5_500.jsonl
position        FOUND -> position_begin_middle_end_500.jsonl
counterfactual  FOUND -> counterfactual_high_confidence_fixed.jsonl


## 4. JSONL utilities

In [3]:
def load_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def append_jsonl(records, path):
    with path.open("a", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def experiment_key(row):
    return (
        row["id"],
        row.get("experiment"),
        row.get("condition"),
    )


def load_completed_keys(path):
    completed = set()

    if not path.exists():
        return completed

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            row = json.loads(line)
            completed.add((
                row["id"],
                row.get("experiment"),
                row.get("condition"),
            ))

    return completed

## 5. Load and inspect datasets

In [4]:
clean = load_jsonl(CLEAN_FILE)
distractor = load_jsonl(DISTRACTOR_FILE)
position = load_jsonl(POSITION_FILE)
counterfactual = load_jsonl(COUNTERFACTUAL_FILE)

print("Clean:", len(clean))
print("Distractor:", len(distractor))
print("Position:", len(position))
print("Counterfactual:", len(counterfactual))

print("\nDistractor conditions:")
print(Counter(r["condition"] for r in distractor))

print("\nPosition conditions:")
print(Counter(r["condition"] for r in position))

Clean: 500
Distractor: 1500
Position: 1500
Counterfactual: 431

Distractor conditions:
Counter({'D1': 500, 'D3': 500, 'D5': 500})

Position conditions:
Counter({'beginning': 500, 'middle': 500, 'end': 500})


## 6. Fixed benchmark prompt

In [5]:
SYSTEM_PROMPT = (
    "You are a question-answering system.\n\n"
    "Use only the provided context to answer the question.\n\n"
    "Return only the short answer.\n"
    "Do not provide explanations, reasoning, citations, greetings, or extra text.\n"
    "If the context does not support an answer, return INSUFFICIENT_EVIDENCE."
)


def build_user_prompt(row):
    return (
        "Context:\n"
        + row["context"]
        + "\n\nQuestion:\n"
        + row["question"]
        + "\n\nAnswer:"
    )

## 7. Load vLLM

In [6]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

assert torch.cuda.is_available(), (
    "CUDA GPU not detected. Switch the runtime to an NVIDIA T4 before continuing."
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

llm = LLM(
    model=MODEL_NAME,
    dtype="float16",
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    max_model_len=MAX_MODEL_LEN,
    trust_remote_code=True,
    seed=SEED,
)

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_NEW_TOKENS,
    seed=SEED,
)

print("Model loaded.")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Parse safetensors files:   0%|          | 0/3 [00:00<?, ?it/s]

(MainProcess pid=2506) INFO 10-06 21:53:54 [kernel.py:408] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

(MainProcess pid=2506) WARNING 10-06 21:53:59 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(MainProcess pid=2506) INFO 10-06 22:01:07 [hf.py:646] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
Model loaded.


## 8. Build chat-formatted prompts

In [7]:
def build_chat_prompt(row):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": build_user_prompt(row),
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

## 9. Output cleaner

In [8]:
def clean_prediction(text):
    text = str(text).strip()

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if not lines:
        return ""

    prediction = lines[0]

    prefixes = [
        "Answer:",
        "answer:",
        "Final answer:",
        "final answer:",
    ]

    for prefix in prefixes:
        if prediction.startswith(prefix):
            prediction = prediction[len(prefix):].strip()

    return prediction

## 10. Resumable batched inference function

In [9]:
def run_dataset(
    records,
    output_path,
    dataset_name,
    batch_size=INFERENCE_BATCH_SIZE,
):
    completed = load_completed_keys(output_path)

    pending = [
        row
        for row in records
        if experiment_key(row) not in completed
    ]

    print("=" * 90)
    print("Dataset:", dataset_name)
    print("Total rows:", len(records))
    print("Already completed:", len(records) - len(pending))
    print("Remaining:", len(pending))

    if not pending:
        print("Nothing to do.")
        return

    for start in tqdm(
        range(0, len(pending), batch_size),
        desc=dataset_name
    ):
        batch = pending[start:start + batch_size]

        prompts = [
            build_chat_prompt(row)
            for row in batch
        ]

        outputs = llm.generate(
            prompts,
            sampling_params,
            use_tqdm=False,
        )

        saved = []

        for row, output in zip(batch, outputs):
            raw_prediction = output.outputs[0].text if output.outputs else ""
            prediction = clean_prediction(raw_prediction)

            result = {
                "id": row["id"],
                "experiment": row.get("experiment"),
                "condition": row.get("condition"),
                "question": row["question"],
                "gold_answer": row["gold_answer"],
                "model_answer": prediction,
                "raw_model_output": raw_prediction,
                "model": MODEL_NAME,
                "temperature": TEMPERATURE,
                "max_new_tokens": MAX_NEW_TOKENS,
                "seed": SEED,
            }

            if "num_distractors" in row:
                result["num_distractors"] = row["num_distractors"]

            if "gold_indices" in row:
                result["gold_indices"] = row["gold_indices"]

            if "counterfactual_answer" in row:
                result["counterfactual_answer"] = row["counterfactual_answer"]

            if "counterfactual_type" in row:
                result["counterfactual_type"] = row["counterfactual_type"]

            if "replacement_source" in row:
                result["replacement_source"] = row["replacement_source"]

            saved.append(result)

        append_jsonl(saved, output_path)

    print("Saved:", output_path.resolve())

## 11. Run clean baseline first

In [10]:
run_dataset(
    clean,
    CLEAN_OUT,
    "clean"
)

Dataset: clean
Total rows: 500
Already completed: 500
Remaining: 0
Nothing to do.


## 12. Inspect clean predictions

In [11]:
clean_results = load_jsonl(CLEAN_OUT)

for row in clean_results[:10]:
    print("=" * 100)
    print("Q:", row["question"])
    print("Gold:", row["gold_answer"])
    print("Model:", row["model_answer"])

Q: Paul DeBoy is known for an appearance in the Western action-adventure game developed by which company?
Gold: Rockstar San Diego
Model: Rockstar Games
Q: In what year did the manager of the singer-songwriter best known for "American Pie" die?
Gold: 2005
Model: 2005
Q: Which TV series directed and written by the same person as "Buffy the Vampire Slayer" starred Tahmoh Penikett?
Gold: Dollhouse
Model: Dollhouse
Q: A guitar player called Noodles is a longtime member of which band, The Smashing Pumpkins or The Offspring?
Gold: The Offspring
Model: The Offspring
Q: What is the nickname of the peninsula that is home to the Calabrian Black Squirrel?
Gold: "lo Stivale" (the Boot)
Model: lo Stivale
Q: What aviator participated to the Transatlantic flight organized by the "heir apparent" to Italian dictator Benito Mussolini?
Gold: Antonio Lippi
Model: Antonio Lippi
Q: Which player won more Grand Slam titles, Kevin Ullyett or Billie Jean King?
Gold: Billie Jean King
Model: Billie Jean King won 

## 13. Run distractor experiment

In [12]:
run_dataset(
    distractor,
    DISTRACTOR_OUT,
    "distractor"
)

Dataset: distractor
Total rows: 1500
Already completed: 0
Remaining: 1500


distractor:   0%|          | 0/24 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/rag_stress/outputs/distractor_answers.jsonl


## 14. Run position experiment

In [13]:
run_dataset(
    position,
    POSITION_OUT,
    "position"
)

Dataset: position
Total rows: 1500
Already completed: 1500
Remaining: 0
Nothing to do.


## 15. Run counterfactual experiment

In [14]:
run_dataset(
    counterfactual,
    COUNTERFACTUAL_OUT,
    "counterfactual"
)

Dataset: counterfactual
Total rows: 431
Already completed: 431
Remaining: 0
Nothing to do.


## 16. Verify final output counts

In [15]:
output_pairs = [
    ("clean", CLEAN_OUT, len(clean)),
    ("distractor", DISTRACTOR_OUT, len(distractor)),
    ("position", POSITION_OUT, len(position)),
    ("counterfactual", COUNTERFACTUAL_OUT, len(counterfactual)),
]

all_counts_ok = True

for name, path, expected in output_pairs:
    actual = len(load_jsonl(path)) if path.exists() else 0
    ok = actual == expected
    all_counts_ok &= ok

    print(
        f"{name:15s}",
        f"{actual}/{expected}",
        "OK" if ok else "Error"
    )

print()

if all_counts_ok:
    print("ALL INFERENCE FILES COMPLETE")
else:
    print(
        "Some outputs are incomplete. "
        "Rerun the corresponding cell; completed rows will be skipped."
    )

clean           500/500 OK
distractor      1500/1500 OK
position        1500/1500 OK
counterfactual  431/431 OK

ALL INFERENCE FILES COMPLETE


## 17. Inspect output condition counts

In [16]:
if DISTRACTOR_OUT.exists():
    dr = load_jsonl(DISTRACTOR_OUT)
    print("Distractor:", Counter(r["condition"] for r in dr))

if POSITION_OUT.exists():
    pr = load_jsonl(POSITION_OUT)
    print("Position:", Counter(r["condition"] for r in pr))

Distractor: Counter({'D1': 500, 'D3': 500, 'D5': 500})
Position: Counter({'beginning': 500, 'middle': 500, 'end': 500})


## 18. Duplicate-key safety check

In [17]:
def duplicate_keys(path):
    rows = load_jsonl(path)

    keys = [
        (
            r["id"],
            r["experiment"],
            r["condition"],
        )
        for r in rows
    ]

    counts = Counter(keys)

    return [
        key
        for key, count in counts.items()
        if count > 1
    ]


for path in [
    CLEAN_OUT,
    DISTRACTOR_OUT,
    POSITION_OUT,
    COUNTERFACTUAL_OUT,
]:
    if not path.exists():
        continue

    dupes = duplicate_keys(path)

    print(
        path.name,
        "duplicate keys:",
        len(dupes)
    )

clean_answers.jsonl duplicate keys: 0
distractor_answers.jsonl duplicate keys: 0
position_answers.jsonl duplicate keys: 0
counterfactual_answers.jsonl duplicate keys: 0
